# DRC Violation task: BiasedLoss + GroupNorm seed sweep (merged)

Merged version of `show_experiment_result.ipynb`: every quantity is aggregated over the seeds of `run_sweep_seed.sh` (mean, and std with ddof=1 across seeds).

Run it from the directory that contains the `.aim` repo. Runs are recognised by `save_path` (contains `sweep_seed` and `_seed<N>`), so no manual tags are needed. If a seed was run twice, the most recent run is kept.

### Baseline configuration
* max iterations: 40000 (see the sweep configs), in/out channels: 9, 1
* threshold: 0.1 (used to build the confusion matrix)

In [ ]:
# Imports
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aim import Repo
from aim.storage.context import Context

repo = Repo(path=os.getcwd())

TRAIN_EXPERIMENT = 'drc_centralized_train_corrected'
TEST_EXPERIMENT = 'drc_centralized_test_corrected'
SWEEP_MARK = 'sweep_seed'
SEED_RE = re.compile(r'_seed(\d+)')
FPR_GRID = np.linspace(0, 1, 501)

def get_sweep_runs(experiment_name):
    """{seed: Run} for the sweep, keeping the most recent finished run per seed."""
    query = f'run.active == False and run.archived == False and run.experiment == "{experiment_name}"'
    found = {}
    for wrapper in repo.query_runs(query=query, report_mode=0).iter_runs():
        run = wrapper.run
        hparams = run.get('hparams')
        save_path = str(hparams['save_path']) if hparams is not None else ''
        m = SEED_RE.search(save_path)
        if SWEEP_MARK not in save_path or m is None:
            continue
        seed = int(m.group(1))
        if seed not in found or run.creation_time > found[seed].creation_time:
            found[seed] = run
    return dict(sorted(found.items()))

def return_metrics_sorted(run, name, context):
    metric = run.get_metric(name, context=Context(context))
    if metric is None:
        return None
    ts = metric.timestamps.values_numpy()
    return metric.values.values_numpy()[np.argsort(ts)]

def get_single_value_metric(run, name, subset='test'):
    vals = return_metrics_sorted(run, name, {'subset': subset})
    return None if vals is None or len(vals) == 0 else float(vals[0])

def roc_curve(run):
    tpr = return_metrics_sorted(run, 'ROC_TPR', {'type': 'curve'})
    fpr = return_metrics_sorted(run, 'ROC_FPR', {'type': 'curve'})
    if tpr is None or fpr is None:
        return None
    order = np.argsort(fpr, kind='stable')
    return fpr[order], tpr[order]

def auc(fpr, tpr):
    return float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2))

def mean_std(arrs):
    """Point-wise mean / std (ddof=1) over seeds, truncated to the shortest curve."""
    n = min(len(a) for a in arrs)
    stack = np.stack([np.asarray(a)[:n] for a in arrs])
    return stack, stack.mean(0), (stack.std(0, ddof=1) if len(stack) > 1 else np.zeros(n))

train_runs = get_sweep_runs(TRAIN_EXPERIMENT)
test_runs = get_sweep_runs(TEST_EXPERIMENT)
print('train seeds:', list(train_runs), '| test seeds:', list(test_runs))

## Train results
Configuration used by each seed (must be identical apart from the seed)

In [ ]:
rows = []
for seed, run in train_runs.items():
    h = run.get('hparams')
    rows.append({
        'seed': seed,
        'tag': run.tags[0] if len(run.tags) else run.name,
        'batch_size': h['batch_size'],
        'eval_metrics': h['eval_metric'],
        'loss_type': h['loss_type'],
        'lr': h['lr'],
        'threshold': h['threshold'],
        'weight_decay': h['weight_decay'],
        'save_path': h['save_path'],
    })
display(pd.DataFrame(rows))

## Train results graphs (epoch loss and pixel loss per iteration, mean over seeds)

In [ ]:
pixel = {s: return_metrics_sorted(r, 'Pixel Loss', {'subset': 'global'}) for s, r in train_runs.items()}
epoch = {s: return_metrics_sorted(r, 'Epoch Loss', {'subset': 'global'}) for s, r in train_runs.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plt.suptitle(f"BS_GN train, {len(train_runs)} seeds")
for ax, curves, title in ((axes[0], pixel, "Pixel Loss per each iteration"),
                          (axes[1], epoch, "Epoch Loss (avg pixel loss for each epoch)")):
    stack, mean, std = mean_std(list(curves.values()))
    x = np.arange(len(mean))
    for s, c in zip(curves, stack):
        ax.plot(x, c, color='0.75', linewidth=0.7)
    ax.plot(x, mean, color='C0', label='mean over seeds')
    ax.fill_between(x, mean - std, mean + std, color='C0', alpha=0.25, label='+- std')
    ax.scatter(len(mean) - 1, mean[-1])
    ax.annotate(f"Last value: {mean[-1]:.4f} +- {std[-1]:.4f}", (len(mean) - 1, mean[-1]),
                textcoords="offset points", xytext=(-90, 10))
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

loss_df = pd.DataFrame([{'seed': s, 'pixel_loss': pixel[s][-1], 'epoch_loss': epoch[s][-1]} for s in train_runs])
display(loss_df)
display(pd.DataFrame({'mean': loss_df[['pixel_loss', 'epoch_loss']].mean(),
                      'std': loss_df[['pixel_loss', 'epoch_loss']].std(ddof=1)}))

### Test results distribution (metrics)
Per-sample NRMS / SSIM, pooled over all seeds (the mean of the per-seed means is in the final table).

In [ ]:
dist = {'NRMS': [], 'SSIM': []}
for seed, run in test_runs.items():
    for name in dist:
        v = return_metrics_sorted(run, f'Test {name} (dist)', {'subset': 'test', 'aggregation': 'distribution'})
        dist[name].append(np.array([]) if v is None else v)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plt.suptitle(f"BS_GN test, {len(test_runs)} seeds pooled")
for ax, name in zip(axes, dist):
    pooled = np.concatenate(dist[name]) if dist[name] else np.array([])
    ax.hist(pooled, bins=30, edgecolor='black', alpha=0.7)
    ax.set_title(f"{name} distribution")
    ax.set_ylabel(name)
    if len(pooled):
        ax.text(0.98, 0.98, f"mean={pooled.mean():.4f}\nstd={pooled.std():.4f}",
                transform=ax.transAxes, ha='right', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='0.7'))
plt.show()

### Test results distribution (ROC)
Every seed is interpolated on a common FPR grid, then averaged point-wise.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
interp = []
for seed, run in test_runs.items():
    curve = roc_curve(run)
    if curve is None:
        continue
    fpr, tpr = curve
    uniq, idx = np.unique(fpr, return_index=True)
    t = np.interp(FPR_GRID, uniq, tpr[idx])
    interp.append(t)
    ax.plot(FPR_GRID, t, color='0.7', linewidth=0.8, label=f"seed {seed} (AUC={auc(fpr, tpr):.4f})")

if interp:
    stack, mean, std = mean_std(interp)
    aucs = [auc(FPR_GRID, t) for t in stack]
    ax.plot(FPR_GRID, mean, color='C0', label=f"mean (AUC={np.mean(aucs):.4f} +- {np.std(aucs, ddof=1) if len(aucs) > 1 else 0:.4f})")
    ax.fill_between(FPR_GRID, mean - std, mean + std, color='C0', alpha=0.25)

ax.plot([0, 1], [0, 1], linestyle='--', color='k')
ax.set_title("ROC Curves (mean +- std over seeds)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Test confusion matrix (aggregated per run, averaged over seeds)

Each seed's TP/TN/FP/FN is converted to fractions of the total test pixels, then averaged over seeds. Cells show mean +- std.

In [ ]:
cms, raw = [], []
for seed, run in test_runs.items():
    vals = [get_single_value_metric(run, f'Confusion {k}') for k in ('TN', 'FP', 'FN', 'TP')]
    if None in vals:
        continue
    cm = np.array(vals, dtype=float).reshape(2, 2)
    raw.append(cm)
    cms.append(cm / cm.sum())

if cms:
    mean_cm = np.mean(cms, axis=0)
    std_cm = np.std(cms, axis=0, ddof=1) if len(cms) > 1 else np.zeros((2, 2))
    print(f"TP: {mean_cm[1,1]}\nTN: {mean_cm[0,0]}\nFP: {mean_cm[0,1]}\nFN: {mean_cm[1,0]}")

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    im = ax.imshow(mean_cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1']); ax.set_yticklabels(['True 0', 'True 1'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Mean confusion matrix ({len(cms)} seeds)')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{100*mean_cm[i,j]:.2f}%\n+- {100*std_cm[i,j]:.2f}', ha='center', va='center',
                    color='white' if mean_cm[i, j] > mean_cm.max() / 2 else 'black', fontsize=11)

    # accuracy / precision / recall per seed -> mean +- std
    pr = np.array([[ (c[0,0]+c[1,1]) / c.sum(),
                     c[1,1] / (c[1,1]+c[0,1]) if (c[1,1]+c[0,1]) else 0.0,
                     c[1,1] / (c[1,1]+c[1,0]) if (c[1,1]+c[1,0]) else 0.0] for c in raw])
    m, s = pr.mean(0), (pr.std(0, ddof=1) if len(pr) > 1 else np.zeros(3))
    ax.text(1.55, 0.5, f'Accuracy: {m[0]:.4f} +- {s[0]:.4f}\nPrecision: {m[1]:.4f} +- {s[1]:.4f}\nRecall: {m[2]:.4f} +- {s[2]:.4f}',
            transform=ax.transAxes, va='center', ha='left', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='0.7'))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

### Test final table results
One row per seed, followed by mean and std (ddof=1) over seeds.

In [ ]:
table = []
for seed, run in test_runs.items():
    row = {'seed': seed}
    for name in ('NRMS', 'SSIM'):
        v = return_metrics_sorted(run, f'Test {name} (dist)', {'subset': 'test', 'aggregation': 'distribution'})
        row[f'Avg {name}'] = float(np.mean(v)) if v is not None and len(v) else None
    curve = roc_curve(run)
    if curve is not None:
        fpr, tpr = curve
        row['AUC ROC'] = auc(fpr, tpr)
        valid = np.where(fpr <= 0.05)[0]
        if len(valid):
            row['FPR <= 5%'] = float(fpr[valid[-1]])
            row['TPR @ FPR=5%'] = float(tpr[valid[-1]])
    row['accuracy'] = get_single_value_metric(run, 'Test Accuracy')
    row['precision'] = get_single_value_metric(run, 'Test Precision')
    cm = [get_single_value_metric(run, f'Confusion {k}') for k in ('TN', 'FP', 'FN', 'TP')]
    if None not in cm:
        tn, fp, fn, tp = cm
        row['recall'] = tp / (tp + fn) if (tp + fn) else 0.0
    table.append(row)

final_df = pd.DataFrame(table)
display(final_df)

metric_cols = [c for c in final_df.columns if c != 'seed']
summary = pd.DataFrame({'mean': final_df[metric_cols].mean(), 'std': final_df[metric_cols].std(ddof=1)})
summary['mean +- std'] = summary.apply(lambda r: f"{r['mean']:.4f} +- {r['std']:.4f}", axis=1)
display(summary)